# 03 Levene Check

4群の分散が同じくらいかを確認するNotebook。

- 入力: `data/processed/{DATASET_NAME}/team_centroids_by_sequence.csv`
- 必須列: `group`, `mean_centroid_x`, `mean_centroid_y`
- 有意水準: `alpha = 0.05`

In [ ]:
from pathlib import Path

import pandas as pd
from scipy.stats import levene

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name in ('重心', 'effective_area') else Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MATCH_DATE = '2023-11-25'
MATCH_ID = '117093'
MATCH_LABEL = 'tsukuba_vs_tsukuba-b'
DATASET_NAME = f'{MATCH_DATE}_{MATCH_ID}_{MATCH_LABEL}'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed' / DATASET_NAME

INPUT_PATH = PROCESSED_DIR / 'team_centroids_by_sequence.csv'

ALPHA = 0.05
REQUIRED_COLUMNS = ['group', 'mean_centroid_x', 'mean_centroid_y']


In [ ]:
def load_centroid_sequences(path):
    if not path.exists():
        raise FileNotFoundError(f'重心分析用CSVが見つかりません: {path}')
    df = pd.read_csv(path)
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f'必須列が不足しています: {missing}')
    return df


def levene_across_groups(df, value_col, group_col='group', alpha=ALPHA):
    samples = [sub[value_col].dropna().to_numpy() for _, sub in df.groupby(group_col)]
    statistic, p_value = levene(*samples)
    return pd.DataFrame([{
        'test': 'levene',
        'variable': value_col,
        'group': 'all',
        'n': sum(len(sample) for sample in samples),
        'statistic': statistic,
        'p_value': p_value,
        'alpha': alpha,
        'passed': bool(p_value > alpha),
    }])

In [ ]:
df = load_centroid_sequences(INPUT_PATH)
df.head()

## x 方向

In [ ]:
levene_x = levene_across_groups(df, 'mean_centroid_x')
levene_x

## y 方向

In [ ]:
levene_y = levene_across_groups(df, 'mean_centroid_y')
levene_y